## Notebook 2: Preprocessing
### Merge all cleaned data into one master dataframe
### Handle missing data and outliers

____________

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

# Load all cleaned CSVs
df_pam          = pd.read_csv(PROCESSED_DIR / "pam.csv")
df_activity     = pd.read_csv(PROCESSED_DIR / "activity.csv")
df_dark         = pd.read_csv(PROCESSED_DIR / "dark.csv")
df_conversation = pd.read_csv(PROCESSED_DIR / "conversation.csv")
df_gps          = pd.read_csv(PROCESSED_DIR / "gps.csv")
df_deadlines    = pd.read_csv(PROCESSED_DIR / "deadlines.csv")

# Standardise all date columns to tz-naive
def clean_date(df):
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None).dt.normalize()
    return df

df_pam          = clean_date(df_pam)
df_activity     = clean_date(df_activity)
df_dark         = clean_date(df_dark)
df_conversation = clean_date(df_conversation)
df_gps          = clean_date(df_gps)
df_deadlines    = clean_date(df_deadlines)

# Merge everything — start with PAM as base
df = df_pam.copy()
df = df.merge(df_activity,     on=["hashed_uid", "date"], how="outer")
df = df.merge(df_dark,         on=["hashed_uid", "date"], how="outer")
df = df.merge(df_conversation, on=["hashed_uid", "date"], how="outer")
df = df.merge(df_gps,          on=["hashed_uid", "date"], how="outer")
df = df.merge(df_deadlines,    on=["hashed_uid", "date"], how="outer")

df = df.sort_values(["hashed_uid", "date"]).reset_index(drop=True)

print(f"Merged shape: {df.shape}")
print(f"Participants: {df['hashed_uid'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

# Find rows without PAM
missing_pam = df[df["pam_score_mean"].isna()]
print(f"\nRows without PAM score: {len(missing_pam)}")
print(f"Rows with PAM score: {df['pam_score_mean'].notna().sum()}")

Merged shape: (3339, 13)
Participants: 49
Date range: 2013-03-25 to 2013-06-03

Rows without PAM score: 1088
Rows with PAM score: 2251


In [2]:
# Drop rows where PAM is missing — these cannot be used in analysis
# since PAM is our outcome variable
df = df.dropna(subset=["pam_score_mean"])

print(f"After dropping missing PAM: {df.shape}")
print(f"Participants remaining: {df['hashed_uid'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nMissing rates per column:")
for col in df.columns:
    rate = df[col].isna().mean()
    if rate > 0:
        print(f"  {col}: {rate:.1%}")

After dropping missing PAM: (2251, 13)
Participants remaining: 49
Date range: 2013-03-25 to 2013-06-03

Missing rates per column:
  frac_stationary: 4.4%
  n_activity_readings: 4.4%
  dark_hours: 16.9%
  conversation_hours: 9.6%
  n_gps_readings: 5.7%
  mean_accuracy: 5.7%
  km_from_campus: 5.7%
  n_deadlines: 10.7%
  has_deadline: 10.7%


In [3]:
# Weekday vs weekend — derived directly from date column
# Monday=0, Tuesday=1, ..., Saturday=5, Sunday=6
df["is_weekend"] = (df["date"].dt.dayofweek >= 5).astype(int)
df["day_of_week"] = df["date"].dt.day_name()

print("Weekday/weekend flag added.")
print(df["day_of_week"].value_counts())
print(f"\nWeekend days: {df['is_weekend'].sum()} ({df['is_weekend'].mean():.1%})")
print(f"Weekday days: {(df['is_weekend']==0).sum()} ({(df['is_weekend']==0).mean():.1%})")

Weekday/weekend flag added.
day_of_week
Wednesday    342
Monday       331
Friday       326
Thursday     323
Tuesday      314
Sunday       310
Saturday     305
Name: count, dtype: int64

Weekend days: 615 (27.3%)
Weekday days: 1636 (72.7%)


In [4]:
df.to_csv(PROCESSED_DIR / "master.csv", index=False)

print("Master saved.")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Master saved.
Shape: (2251, 15)
Columns: ['hashed_uid', 'date', 'pam_score_mean', 'pam_n_responses', 'frac_stationary', 'n_activity_readings', 'dark_hours', 'conversation_hours', 'n_gps_readings', 'mean_accuracy', 'km_from_campus', 'n_deadlines', 'has_deadline', 'is_weekend', 'day_of_week']


In [5]:
df

,hashed_uid,date,pam_score_mean,pam_n_responses,frac_stationary,n_activity_readings,dark_hours,conversation_hours,n_gps_readings,mean_accuracy,km_from_campus,n_deadlines,has_deadline,is_weekend,day_of_week
1,02b34100f41b4332,2013-03-27,9.400000,5.0,0.922105,8229.0,16.398889,4.809444,71.0,32.826831,0.177483,0.0,0.0,0,Wednesday
2,02b34100f41b4332,2013-03-28,10.250000,4.0,0.946593,8276.0,11.093056,4.365000,71.0,25.101197,0.178181,0.0,0.0,0,Thursday
3,02b34100f41b4332,2013-03-29,6.000000,3.0,0.943575,8241.0,16.166111,6.332222,71.0,25.993324,0.179254,0.0,0.0,0,Friday
4,02b34100f41b4332,2013-03-30,9.200000,5.0,0.994150,8205.0,13.921667,1.251667,72.0,24.806097,0.180821,0.0,0.0,1,Saturday
5,02b34100f41b4332,2013-03-31,7.000000,2.0,1.000000,8319.0,8.480556,0.759167,72.0,23.304792,0.179454,0.0,0.0,1,Sunday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3334,f87713456680124e,2013-05-30,6.800000,5.0,0.850947,8185.0,10.143889,9.363611,87.0,28.890195,0.288053,0.0,0.0,0,Thursday
3335,f87713456680124e,2013-05-31,5.500000,4.0,0.891838,8025.0,8.286944,6.536111,31.0,37.948161,0.264424,0.0,0.0,0,Friday
3336,f87713456680124e,2013-06-01,3.000000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1,Saturday
3337,f87713456680124e,2013-06-02,6.000000,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1,Sunday


### Weather data adding

In [7]:
PROCESSED_DIR = Path("../data/processed")

# Load weather CSV
df_weather = pd.read_csv(PROCESSED_DIR / "weather.csv")

# Ensure both date columns are timezone-naive daily timestamps
df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()
df_weather["date"] = pd.to_datetime(df_weather["date"]).dt.tz_localize(None).dt.normalize()

# Merge weather into main dataframe
df = df.merge(
    df_weather,
    on=["hashed_uid", "date"],
    how="left"
)

print(df.head())
print(df.columns.tolist())

         hashed_uid       date  pam_score_mean  pam_n_responses  \
0  02b34100f41b4332 2013-03-27            9.40              5.0   
1  02b34100f41b4332 2013-03-28           10.25              4.0   
2  02b34100f41b4332 2013-03-29            6.00              3.0   
3  02b34100f41b4332 2013-03-30            9.20              5.0   
4  02b34100f41b4332 2013-03-31            7.00              2.0   

   frac_stationary  n_activity_readings  dark_hours  conversation_hours  \
0         0.922105               8229.0   16.398889            4.809444   
1         0.946593               8276.0   11.093056            4.365000   
2         0.943575               8241.0   16.166111            6.332222   
3         0.994150               8205.0   13.921667            1.251667   
4         1.000000               8319.0    8.480556            0.759167   

   n_gps_readings  mean_accuracy  ...  has_deadline  is_weekend  day_of_week  \
0            71.0      32.826831  ...           0.0           0   

In [8]:
df

,hashed_uid,date,pam_score_mean,pam_n_responses,frac_stationary,n_activity_readings,dark_hours,conversation_hours,n_gps_readings,mean_accuracy,...,has_deadline,is_weekend,day_of_week,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max
0,02b34100f41b4332,2013-03-27,9.400000,5.0,0.922105,8229.0,16.398889,4.809444,71.0,32.826831,...,0.0,0,Wednesday,71.0,0.9,7.2,-6.1,0.3,0.21,17.7
1,02b34100f41b4332,2013-03-28,10.250000,4.0,0.946593,8276.0,11.093056,4.365000,71.0,25.101197,...,0.0,0,Thursday,71.0,3.5,8.0,0.9,1.0,0.63,14.4
2,02b34100f41b4332,2013-03-29,6.000000,3.0,0.943575,8241.0,16.166111,6.332222,71.0,25.993324,...,0.0,0,Friday,53.0,3.4,8.6,-1.6,1.6,0.00,16.6
3,02b34100f41b4332,2013-03-30,9.200000,5.0,0.994150,8205.0,13.921667,1.251667,72.0,24.806097,...,0.0,1,Saturday,3.0,3.0,9.0,-2.6,0.0,0.00,15.1
4,02b34100f41b4332,2013-03-31,7.000000,2.0,1.000000,8319.0,8.480556,0.759167,72.0,23.304792,...,0.0,1,Sunday,51.0,5.3,10.9,-1.7,0.1,0.00,18.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2246,f87713456680124e,2013-05-30,6.800000,5.0,0.850947,8185.0,10.143889,9.363611,87.0,28.890195,...,0.0,0,Thursday,51.0,20.5,26.8,13.0,0.2,0.00,17.2
2247,f87713456680124e,2013-05-31,5.500000,4.0,0.891838,8025.0,8.286944,6.536111,31.0,37.948161,...,0.0,0,Friday,2.0,22.8,30.0,14.1,0.0,0.00,11.8
2248,f87713456680124e,2013-06-01,3.000000,4.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,1,Saturday,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2249,f87713456680124e,2013-06-02,6.000000,6.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,1,Sunday,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# ─────────────────────────────────────────────────────────────────
# WEATHER CATEGORY (derived feature)
# ─────────────────────────────────────────────────────────────────

def map_weather(code):
    if pd.isna(code):
        return "unknown"
    elif code == 0:
        return "clear"
    elif code in [1, 2, 3]:
        return "cloudy"
    elif code in range(51, 68) or code in range(80, 83):
        return "rain"
    elif code in range(71, 78) or code in range(85, 87):
        return "snow"
    elif code >= 95:
        return "storm"
    else:
        return "other"

df["weather_category"] = df["weather_code"].apply(map_weather)

print("Weather categories added:")
print(df["weather_category"].value_counts())

Weather categories added:
weather_category
cloudy     963
rain       729
snow       311
unknown    128
clear      120
Name: count, dtype: int64
